# Finetuning with Distillation and Pruning

This notebook demonstrates how to finetune a GlinerMinus model using:
1. **Knowledge distillation** – a teacher model guides the student via soft logits
2. **Iterative layer pruning** – redundant encoder layers are removed based on angular distance analysis

The workflow: finetune → run layer analysis → prune the most redundant layer → repeat until the target layer count is reached.

In [1]:
# Configuration
BACKBONE = "prajjwal1/bert-mini"  # 4 layers; good for pruning demo (4 → 3 → 2)
OUTPUT_DIR = "./outputs/prune_distill_demo"
TARGET_NUM_LAYERS = 2
PRUNING_EPOCHS_PER_PHASE = 0.5  # Short phases for demo
MAX_STEPS = 50  # Cap total steps for quick run
BATCH_SIZE = 4

In [2]:
import logging

import torch
from torch.utils.data import Dataset
from transformers import set_seed

from glinerminus import (
    GlinerMinusConfig,
    GlinerMinusDataCollator,
    GlinerMinusModel,
    PruningDistillationTrainer,
    PruningTrainingArguments,
)

logging.basicConfig(level=logging.INFO)
set_seed(42)

/home/cheong/Desktop/Work/glinerminus/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [3]:
class SampleGlinerDataset(Dataset):
    """Synthetic dataset for GlinerMinus finetuning.

    Yields entity–span pairs with one-hot style labels:
    0: Match, 1: Neutral, 2: No Match. Uses -100 for ignored positions.
    """

    def __init__(
        self,
        model: GlinerMinusModel,
        n: int = 32,
        entities_per_example: int = 2,
        spans_per_example: int = 3,
    ) -> None:
        self.model = model
        self.n = n
        self.entities_per_example = entities_per_example
        self.spans_per_example = spans_per_example
        # Simple templates for reproducibility
        self.texts = [
            "Alice works at Google in London.",
            "Bob met Eve at the conference in Paris.",
            "The CEO of Microsoft announced new products.",
            "Dr. Smith from Harvard University spoke.",
        ] * (n // 4 + 1)
        self.entity_pools = [
            ["person", "organization", "location"],
            ["person", "organization"],
            ["person", "location"],
        ]

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int) -> dict:
        text = self.texts[idx % len(self.texts)]
        entities = self.entity_pools[idx % len(self.entity_pools)][: self.entities_per_example]
        # Simple character spans (approximate)
        spans = [(0, 4), (14, 19), (23, 28)][: self.spans_per_example]
        inputs = self.model.tokenize(
            [entities],
            [text],
            [spans],
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt",
        )
        out = {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "entity_index": inputs["entity_index"][0],
            "spans": inputs["spans"][0],
        }
        max_e, max_s = len(out["entity_index"]), len(out["spans"])
        labels = torch.full((max_e, max_s, 3), -100.0)
        for e in range(max_e):
            for s in range(max_s):
                labels[e, s, 1] = 1.0  # Neutral for demo
        out["labels"] = labels
        return out

In [4]:
# Create config and models
config = GlinerMinusConfig(
    backbone_model_name_or_path=BACKBONE,
    num_bilinear_heads=3,
)
student = GlinerMinusModel(config)
teacher = GlinerMinusModel(config)
# Copy teacher from student so teacher has trained head; then we keep teacher fixed
teacher.load_state_dict(student.state_dict(), strict=True)
# Teacher stays full; student will be pruned down to TARGET_NUM_LAYERS
print(f"Backbone layers: {student.backbone.config.num_hidden_layers}")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Backbone layers: 4


In [5]:
# Build datasets and dataloaders
train_dataset = SampleGlinerDataset(student, n=64, entities_per_example=2, spans_per_example=3)
collator = GlinerMinusDataCollator()

In [6]:
# Pruning dataloader: layer analysis needs only input_ids and attention_mask.
# We sample from the training data (tokenized text is sufficient).
from torch.utils.data import DataLoader, Subset


def extract_pruning_batch(features: list) -> dict:
    """Extract input_ids and attention_mask for layer analysis."""
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
    }


pruning_dataset = Subset(train_dataset, range(min(32, len(train_dataset))))
pruning_dataloader = DataLoader(
    pruning_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: extract_pruning_batch(batch),
)

In [7]:
# Training arguments: distillation + iterative pruning
args = PruningTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    # Pruning
    target_num_layers=TARGET_NUM_LAYERS,
    pruning_epochs_per_phase=PRUNING_EPOCHS_PER_PHASE,
    pruning_steps_per_phase=-1,
    # Distillation (teacher provided explicitly; these are defaults)
    distillation_temperature=2.0,
    distillation_alpha=0.5,
)

In [8]:
# Run distillation + pruning training
trainer = PruningDistillationTrainer(
    model=student,
    args=args,
    teacher_model=teacher,
    train_dataset=train_dataset,
    data_collator=collator,
    pruning_dataloader=pruning_dataloader,
)
trainer.train()

INFO:glinerminus.training:Phase 1: training with 4 layers (target: 2) for 8 steps.


Step,Training Loss
5,106.472000


INFO:glinerminus.layer_pruning:Layer 2 to 3 has minimum average distance 0.1233. Consider pruning this block.
INFO:glinerminus.training:Pruning layer 2 (1-based).
INFO:glinerminus.training:Phase 2: training with 3 layers (target: 2) for 8 steps.


Step,Training Loss
5,109.179700


INFO:glinerminus.layer_pruning:Layer 1 to 2 has minimum average distance 0.1466. Consider pruning this block.
INFO:glinerminus.training:Pruning layer 1 (1-based).


In [9]:
# Inspect result
final_layers = student.backbone.config.num_hidden_layers
print(f"Final backbone layers: {final_layers} (target was {TARGET_NUM_LAYERS})")
trainer.save_model(OUTPUT_DIR)

Final backbone layers: 2 (target was 2)
